# OAI MRI 3D CNN - Intensity-Based Slice Selection

Clean Kaggle notebook flow for the 3D CNN experiment requested by ma'am:

- Load OAI-MRI-3DDESS `.npy` files
- Split at MRI-volume level to avoid data leakage
- Select only the top 40 or 50 informative slices using intensity variation
- Apply volume normalization and training augmentation
- Handle class imbalance using `pos_weight`
- Fine-tune a pretrained MedicalNet 3D ResNet-18 and save best weights
- Report network parameters, training history, confusion matrix, ROC curve, and final metrics


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Main experiment settings
TOP_K_SLICES = 50        # Next run: use 50 slices. The 40-slice run reached Test AUC ~0.7957.
BATCH_SIZE_3D = 2        # Keep 2 for 3D CNN on Kaggle T4/P100. Use 1 if GPU memory error occurs.
NUM_WORKERS = 2
EPOCHS = 50
PATIENCE = 7
BACKBONE_LR = 2e-5     # Gentle fine-tuning for pretrained MedicalNet layers.
HEAD_LR = 1e-4         # Faster learning for the newly initialized binary classifier head.
WEIGHT_DECAY = 5e-4
ACCUMULATION_STEPS = 4   # Effective batch size = BATCH_SIZE_3D * ACCUMULATION_STEPS
AUGMENT_FLIP_PROB = 0.5
AUGMENT_AFFINE_PROB = 0.5
MAX_ROTATION_DEGREES = 7.0
MAX_SHIFT_PIXELS = 5
MEDICALNET_MODEL_NAME = "MedicalNet-ResNet18-23datasets"
MEDICALNET_WEIGHTS_URL = "https://huggingface.co/TencentMedicalNet/MedicalNet-Resnet18/resolve/main/resnet_18_23dataset.pth"
MEDICALNET_WEIGHTS_PATH = Path("/kaggle/working/resnet_18_23dataset.pth")

OUTPUT_DIR = Path("/kaggle/working")
BEST_MODEL_PATH = OUTPUT_DIR / f"intensity_selected_medicalnet_resnet18_top{TOP_K_SLICES}_best.pth"
HISTORY_CSV_PATH = OUTPUT_DIR / f"intensity_selected_medicalnet_resnet18_top{TOP_K_SLICES}_history.csv"
RESULTS_CSV_PATH = OUTPUT_DIR / f"intensity_selected_medicalnet_resnet18_top{TOP_K_SLICES}_results.csv"

print("Top-K slices:", TOP_K_SLICES)
print("Batch size:", BATCH_SIZE_3D)
print("Backbone LR:", BACKBONE_LR)
print("Head LR:", HEAD_LR)
print("Training epochs (maximum):", EPOCHS)
print(f"Augmentation: horizontal flip, rotation +/-{MAX_ROTATION_DEGREES} degrees, shift +/-{MAX_SHIFT_PIXELS} px")
print("Pretrained model:", MEDICALNET_MODEL_NAME)
print("Best model path:", BEST_MODEL_PATH)


In [ ]:
# Robust Kaggle dataset discovery. This avoids path errors if Kaggle mounts the dataset differently.
INPUT_DIR = Path("/kaggle/input")
normal_files = list(INPUT_DIR.rglob("normal-3DESS-128-64.npy"))

if not normal_files:
    raise FileNotFoundError(
        "Could not find normal-3DESS-128-64.npy under /kaggle/input. "
        "Please make sure the OAI-MRI-3DDESS dataset is added to this notebook."
    )

DATASET_DIR = normal_files[0].parent
normal_path = DATASET_DIR / "normal-3DESS-128-64.npy"
abnormal_path = DATASET_DIR / "abnormal-3DESS-128-64.npy"

normal_data = np.load(normal_path, mmap_mode="r")
abnormal_data = np.load(abnormal_path, mmap_mode="r")

print("Dataset directory:", DATASET_DIR)
print("Normal shape:", normal_data.shape, normal_data.dtype)
print("Abnormal shape:", abnormal_data.shape, abnormal_data.dtype)


In [ ]:
num_normal = normal_data.shape[0]
num_abnormal = abnormal_data.shape[0]
total = num_normal + num_abnormal

normal_labels = np.zeros(num_normal, dtype=np.int64)
abnormal_labels = np.ones(num_abnormal, dtype=np.int64)

all_indices = np.arange(total)
all_labels = np.concatenate([normal_labels, abnormal_labels])

train_indices, temp_indices, train_labels, temp_labels = train_test_split(
    all_indices,
    all_labels,
    test_size=0.30,
    random_state=SEED,
    stratify=all_labels,
)

val_indices, test_indices, val_labels, test_labels = train_test_split(
    temp_indices,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels,
)

assert len(set(train_indices) & set(val_indices)) == 0, "Train/validation leakage detected."
assert len(set(train_indices) & set(test_indices)) == 0, "Train/test leakage detected."
assert len(set(val_indices) & set(test_indices)) == 0, "Validation/test leakage detected."

split_df = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Samples": [len(train_indices), len(val_indices), len(test_indices)],
    "Normal": [
        int((train_labels == 0).sum()),
        int((val_labels == 0).sum()),
        int((test_labels == 0).sum()),
    ],
    "Abnormal": [
        int((train_labels == 1).sum()),
        int((val_labels == 1).sum()),
        int((test_labels == 1).sum()),
    ],
})
split_df["Abnormal %"] = 100 * split_df["Abnormal"] / split_df["Samples"]
split_df["Normal %"] = 100 * split_df["Normal"] / split_df["Samples"]

print("No patient-volume index overlap between train, validation, and test splits.")
print(f"Overall normal/abnormal ratio: {num_normal}:{num_abnormal} = {num_normal / num_abnormal:.3f}:1")

split_df


## Intensity-Based Slice Selection

Each MRI volume has 64 sagittal slices. Instead of using only the middle slice or manually removing fixed slice ranges, this cell scores every slice using intensity variation.

Higher standard deviation usually means the slice contains more anatomical structure and contrast. We select the top `K` slices and sort them back into anatomical order before feeding them to the 3D CNN.


In [ ]:
def select_top_intensity_slices(volume, top_k=40):
    """Return top-K informative slice indices from a 128 x 128 x 64 MRI volume."""
    if volume.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {volume.shape}")

    num_slices = volume.shape[-1]
    if top_k > num_slices:
        raise ValueError(f"top_k={top_k} cannot exceed available slices={num_slices}")

    # Score each slice by intensity variation. More variation usually means richer anatomy.
    slice_scores = volume.astype(np.float32).std(axis=(0, 1))

    selected = np.argsort(slice_scores)[-top_k:]
    selected = np.sort(selected).astype(np.int64)
    return selected, slice_scores

sample_volume = normal_data[0]
selected_slices, slice_scores = select_top_intensity_slices(sample_volume, TOP_K_SLICES)

print("Selected slices:", selected_slices)
print("Number of selected slices:", len(selected_slices))

plt.figure(figsize=(14, 3))
plt.plot(np.arange(len(slice_scores)), slice_scores, marker="o", linewidth=1)
plt.scatter(selected_slices, slice_scores[selected_slices], color="red", label="Selected slices")
plt.xlabel("Slice index")
plt.ylabel("Intensity standard deviation")
plt.title(f"Intensity-Based Slice Selection Example: Top {TOP_K_SLICES} Slices")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"intensity_slice_selection_top{TOP_K_SLICES}.png", dpi=200)
plt.show()


In [ ]:
def show_selected_slices(volume, selected_slices, max_images=10):
    shown = selected_slices[:max_images]
    plt.figure(figsize=(15, 3))
    for i, slice_idx in enumerate(shown):
        plt.subplot(1, len(shown), i + 1)
        plt.imshow(volume[:, :, slice_idx], cmap="gray")
        plt.title(f"S{slice_idx}")
        plt.axis("off")
    plt.suptitle(f"First {len(shown)} Selected Slices in Anatomical Order")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"selected_slice_examples_top{TOP_K_SLICES}.png", dpi=200)
    plt.show()

show_selected_slices(sample_volume, selected_slices, max_images=10)


In [ ]:
class IntensitySelectedVolumeMRIDataset(Dataset):
    def __init__(self, normal_data, abnormal_data, indices, labels, top_k=40, augment=False):
        self.normal_data = normal_data
        self.abnormal_data = abnormal_data
        self.indices = indices
        self.labels = labels.astype(np.float32)
        self.top_k = top_k
        self.augment = augment
        self.num_normal = normal_data.shape[0]
        self.slice_cache = {}

    def __len__(self):
        return len(self.indices)

    def get_volume(self, original_index):
        if original_index < self.num_normal:
            return self.normal_data[original_index]
        return self.abnormal_data[original_index - self.num_normal]

    def normalize_volume(self, volume):
        volume = volume.astype(np.float32)
        return (volume - volume.mean()) / (volume.std() + 1e-8)

    def random_inplane_affine(self, volume):
        angle = np.deg2rad(np.random.uniform(-MAX_ROTATION_DEGREES, MAX_ROTATION_DEGREES))
        shift_x = np.random.uniform(-MAX_SHIFT_PIXELS, MAX_SHIFT_PIXELS)
        shift_y = np.random.uniform(-MAX_SHIFT_PIXELS, MAX_SHIFT_PIXELS)

        height, width, _ = volume.shape
        theta = torch.tensor(
            [[
                [np.cos(angle), -np.sin(angle), 2.0 * shift_x / width],
                [np.sin(angle), np.cos(angle), 2.0 * shift_y / height],
            ]],
            dtype=torch.float32,
        )

        # Treat selected slices as channels so every slice receives exactly the same transform.
        volume_tensor = torch.from_numpy(np.transpose(volume, (2, 0, 1))).unsqueeze(0)
        grid = F.affine_grid(theta, volume_tensor.size(), align_corners=False)
        transformed = F.grid_sample(
            volume_tensor,
            grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )
        return np.transpose(transformed.squeeze(0).numpy(), (1, 2, 0))

    def augment_volume(self, volume):
        # Left-right flip is used; vertical flipping is avoided because it reverses knee anatomy.
        if np.random.rand() < AUGMENT_FLIP_PROB:
            volume = np.flip(volume, axis=1).copy()

        if np.random.rand() < AUGMENT_AFFINE_PROB:
            volume = self.random_inplane_affine(volume)

        if np.random.rand() < 0.3:
            volume = volume + np.random.normal(0, 0.03, size=volume.shape).astype(np.float32)

        if np.random.rand() < 0.3:
            volume = volume * np.random.uniform(0.9, 1.1)

        return volume

    def get_selected_slices(self, original_index, volume):
        cache_key = int(original_index)
        if cache_key not in self.slice_cache:
            selected_slices, _ = select_top_intensity_slices(volume, self.top_k)
            self.slice_cache[cache_key] = selected_slices
        return self.slice_cache[cache_key]

    def __getitem__(self, idx):
        original_index = int(self.indices[idx])
        label = self.labels[idx]

        volume = self.get_volume(original_index)
        selected_slices = self.get_selected_slices(original_index, volume)
        volume = volume[:, :, selected_slices]

        volume = self.normalize_volume(volume)

        if self.augment:
            volume = self.augment_volume(volume)

        # Input volume: 128 x 128 x K -> Conv3D input: 1 x K x 128 x 128
        volume = np.transpose(volume, (2, 0, 1))
        volume = torch.tensor(volume, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(label, dtype=torch.float32)

        return volume, label


In [ ]:
train_dataset_i3d = IntensitySelectedVolumeMRIDataset(
    normal_data, abnormal_data, train_indices, train_labels,
    top_k=TOP_K_SLICES, augment=True,
)

val_dataset_i3d = IntensitySelectedVolumeMRIDataset(
    normal_data, abnormal_data, val_indices, val_labels,
    top_k=TOP_K_SLICES, augment=False,
)

test_dataset_i3d = IntensitySelectedVolumeMRIDataset(
    normal_data, abnormal_data, test_indices, test_labels,
    top_k=TOP_K_SLICES, augment=False,
)

loader_kwargs = {
    "batch_size": BATCH_SIZE_3D,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True

train_loader_i3d = DataLoader(train_dataset_i3d, shuffle=True, **loader_kwargs)
val_loader_i3d = DataLoader(val_dataset_i3d, shuffle=False, **loader_kwargs)
test_loader_i3d = DataLoader(test_dataset_i3d, shuffle=False, **loader_kwargs)

volumes_i3d, labels_i3d = next(iter(train_loader_i3d))

print("Volume batch shape:", volumes_i3d.shape)
print("Label batch shape:", labels_i3d.shape)
print("Volume mean:", volumes_i3d.mean().item())
print("Volume std:", volumes_i3d.std().item())
print("Labels:", labels_i3d)


## Pretrained MedicalNet 3D ResNet-18

This experiment now uses a genuinely pretrained 3D medical-imaging model instead of a custom ResNet trained from scratch. The backbone is MedicalNet ResNet-18, pretrained across 23 medical datasets, and its segmentation head is replaced with a binary classification head for normal vs abnormal knee MRI fine-tuning.

Reference implementation: https://github.com/Tencent/MedicalNet


In [ ]:
class MedicalNetBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, kernel_size=3, stride=stride,
            padding=dilation, dilation=dilation, bias=False,
        )
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3,
            padding=dilation, dilation=dilation, bias=False,
        )
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + residual)


class MedicalNetResNet18Classifier(nn.Module):
    def __init__(self, dropout=0.5):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv3d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, blocks=2)
        self.layer2 = self._make_layer(128, blocks=2, stride=2)
        self.layer3 = self._make_layer(256, blocks=2, stride=1, dilation=2)
        self.layer4 = self._make_layer(512, blocks=2, stride=1, dilation=4)
        self.pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, 1))

    def _make_layer(self, out_channels, blocks, stride=1, dilation=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv3d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels),
            )

        layers = [MedicalNetBasicBlock(self.in_channels, out_channels, stride, dilation, downsample)]
        self.in_channels = out_channels
        layers.extend(MedicalNetBasicBlock(out_channels, out_channels, dilation=dilation) for _ in range(1, blocks))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = torch.flatten(self.pool(x), 1)
        return self.classifier(x).squeeze(1)


def load_medicalnet_pretrained_weights(model, weights_path, weights_url):
    weights_path = Path(weights_path)
    if not weights_path.exists():
        kaggle_weight_files = list(Path("/kaggle/input").rglob(weights_path.name))
        if kaggle_weight_files:
            weights_path = kaggle_weight_files[0]
            print("Using MedicalNet weights from Kaggle input:", weights_path)
        else:
            print("Downloading official MedicalNet ResNet-18 pretrained weights...")
            torch.hub.download_url_to_file(weights_url, str(weights_path), progress=True)

    checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
    checkpoint_state = checkpoint.get("state_dict", checkpoint)
    checkpoint_state = {key.removeprefix("module."): value for key, value in checkpoint_state.items()}

    model_state = model.state_dict()
    pretrained_state = {
        key: value for key, value in checkpoint_state.items()
        if key in model_state and model_state[key].shape == value.shape
    }
    if len(pretrained_state) < 50:
        raise RuntimeError("Too few MedicalNet pretrained layers matched this model architecture.")

    model_state.update(pretrained_state)
    model.load_state_dict(model_state)
    print(f"Loaded {len(pretrained_state)} pretrained MedicalNet tensors from {weights_path.name}.")
    print("The binary classifier head is newly initialized and will be fine-tuned with the backbone.")
    return model


model_i3d = MedicalNetResNet18Classifier(dropout=0.5)
model_i3d = load_medicalnet_pretrained_weights(
    model_i3d, MEDICALNET_WEIGHTS_PATH, MEDICALNET_WEIGHTS_URL,
).to(device)
print(model_i3d)


In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    memory_mb = trainable * 4 / (1024 * 1024)
    return total, trainable, memory_mb

total_params, trainable_params, parameter_memory_mb = count_parameters(model_i3d)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Approx parameter memory MB:", parameter_memory_mb)


In [ ]:
# Class imbalance handling
num_train_normal = int((train_labels == 0).sum())
num_train_abnormal = int((train_labels == 1).sum())

pos_weight_value = num_train_normal / num_train_abnormal
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

criterion_i3d = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

backbone_params = []
head_params = []
for name, param in model_i3d.named_parameters():
    if name.startswith("classifier."):
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer_i3d = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler_i3d = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_i3d,
    mode="max",
    factor=0.5,
    patience=2,
)

print("Train normal volumes:", num_train_normal)
print("Train abnormal volumes:", num_train_abnormal)
print("Positive class weight:", pos_weight_value)
print("Backbone parameters optimized at LR:", BACKBONE_LR)
print("Classifier head parameters optimized at LR:", HEAD_LR)


In [ ]:
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def run_one_epoch_i3d(model, loader, criterion, optimizer=None, accumulation_steps=1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_probs = []

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    for step, (volumes, labels) in enumerate(loader):
        volumes = volumes.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = model(volumes)
                loss = criterion(logits, labels)

            if is_train:
                scaled_loss = loss / accumulation_steps
                scaler.scale(scaled_loss).backward()

                should_step = (step + 1) % accumulation_steps == 0 or (step + 1) == len(loader)
                if should_step:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

        probs = torch.sigmoid(logits.detach())
        total_loss += loss.item() * volumes.size(0)
        all_labels.extend(labels.detach().cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    all_preds = (all_probs >= 0.5).astype(int)

    return {
        "loss": avg_loss,
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "auc": roc_auc_score(all_labels, all_probs),
    }


In [ ]:
history_i3d = []
best_val_auc = 0.0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_one_epoch_i3d(
        model_i3d,
        train_loader_i3d,
        criterion_i3d,
        optimizer_i3d,
        accumulation_steps=ACCUMULATION_STEPS,
    )

    val_metrics = run_one_epoch_i3d(
        model_i3d,
        val_loader_i3d,
        criterion_i3d,
    )

    scheduler_i3d.step(val_metrics["auc"])
    current_backbone_lr = optimizer_i3d.param_groups[0]["lr"]
    current_head_lr = optimizer_i3d.param_groups[1]["lr"]

    history_i3d.append({
        "epoch": epoch,
        "backbone_lr": current_backbone_lr,
        "head_lr": current_head_lr,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "train_f1": train_metrics["f1"],
        "train_auc": train_metrics["auc"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_auc": val_metrics["auc"],
    })

    if val_metrics["auc"] > best_val_auc:
        best_val_auc = val_metrics["auc"]
        epochs_without_improvement = 0
        torch.save(model_i3d.state_dict(), BEST_MODEL_PATH)
        checkpoint_status = "saved best model"
    else:
        epochs_without_improvement += 1
        checkpoint_status = f"no improvement ({epochs_without_improvement}/{PATIENCE})"

    print(
        f"Epoch {epoch:02d} | "
        f"Backbone LR: {current_backbone_lr:.6f} | "
        f"Head LR: {current_head_lr:.6f} | "
        f"Train AUC: {train_metrics['auc']:.4f} | "
        f"Val AUC: {val_metrics['auc']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"{checkpoint_status}"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

history_i3d_df = pd.DataFrame(history_i3d)
history_i3d_df.to_csv(HISTORY_CSV_PATH, index=False)
history_i3d_df


In [ ]:
plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.plot(history_i3d_df["epoch"], history_i3d_df["train_loss"], label="Train Loss")
plt.plot(history_i3d_df["epoch"], history_i3d_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Intensity-Selected MedicalNet ResNet-18: Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(2, 2, 2)
plt.plot(history_i3d_df["epoch"], history_i3d_df["train_auc"], label="Train AUC")
plt.plot(history_i3d_df["epoch"], history_i3d_df["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Intensity-Selected MedicalNet ResNet-18: AUC")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(2, 2, 3)
plt.plot(history_i3d_df["epoch"], history_i3d_df["train_accuracy"], label="Train Accuracy")
plt.plot(history_i3d_df["epoch"], history_i3d_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Intensity-Selected MedicalNet ResNet-18: Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(2, 2, 4)
plt.plot(history_i3d_df["epoch"], history_i3d_df["train_recall"], label="Train Recall")
plt.plot(history_i3d_df["epoch"], history_i3d_df["val_recall"], label="Validation Recall")
plt.xlabel("Epoch")
plt.ylabel("Recall")
plt.title("Intensity-Selected MedicalNet ResNet-18: Recall")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"intensity_selected_medicalnet_resnet18_top{TOP_K_SLICES}_training_history.png", dpi=200)
plt.show()


In [ ]:
def collect_predictions_i3d(model, loader):
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for volumes, labels in loader:
            volumes = volumes.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = model(volumes)
            probs = torch.sigmoid(logits)

            all_labels.extend(labels.detach().cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_probs)

model_i3d.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

val_labels_i3d, val_probs_i3d = collect_predictions_i3d(model_i3d, val_loader_i3d)
test_labels_i3d, test_probs_i3d = collect_predictions_i3d(model_i3d, test_loader_i3d)

reloaded_val_auc = roc_auc_score(val_labels_i3d, val_probs_i3d)
reloaded_test_auc = roc_auc_score(test_labels_i3d, test_probs_i3d)

best_history_row = history_i3d_df.sort_values("val_auc", ascending=False).iloc[0]
print(f"Best saved epoch from history: {int(best_history_row['epoch'])}")
print(f"Best validation AUC from training log: {best_history_row['val_auc']:.6f}")
print(f"Validation AUC after reloading best checkpoint: {reloaded_val_auc:.6f}")
print(f"Independent test AUC: {reloaded_test_auc:.6f}")


In [ ]:
threshold_rows = []

for threshold in np.arange(0.05, 0.96, 0.01):
    val_preds = (val_probs_i3d >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_labels_i3d, val_preds).ravel()

    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)

    threshold_rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(val_labels_i3d, val_preds),
        "precision": precision_score(val_labels_i3d, val_preds, zero_division=0),
        "recall_abnormal": sensitivity,
        "specificity_normal": specificity,
        "f1": f1_score(val_labels_i3d, val_preds, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(val_labels_i3d, val_preds),
        "auc": roc_auc_score(val_labels_i3d, val_probs_i3d),
    })

threshold_i3d_df = pd.DataFrame(threshold_rows)
best_threshold_i3d_df = threshold_i3d_df.sort_values(
    by=["balanced_accuracy", "f1"],
    ascending=False,
).head(10)

best_threshold_i3d_df


In [ ]:
best_row = best_threshold_i3d_df.iloc[0]
BEST_THRESHOLD_I3D = float(best_row["threshold"])

test_preds_i3d = (test_probs_i3d >= BEST_THRESHOLD_I3D).astype(int)

results_i3d = {
    "model": "Intensity-Selected MedicalNet ResNet-18",
    "top_k_slices": TOP_K_SLICES,
    "pretrained_model": MEDICALNET_MODEL_NAME,
    "threshold": BEST_THRESHOLD_I3D,
    "accuracy": accuracy_score(test_labels_i3d, test_preds_i3d),
    "precision": precision_score(test_labels_i3d, test_preds_i3d, zero_division=0),
    "recall": recall_score(test_labels_i3d, test_preds_i3d, zero_division=0),
    "f1": f1_score(test_labels_i3d, test_preds_i3d, zero_division=0),
    "auc": roc_auc_score(test_labels_i3d, test_probs_i3d),
    "parameters": total_params,
    "trainable_parameters": trainable_params,
    "parameter_memory_mb": parameter_memory_mb,
}

results_i3d_df = pd.DataFrame([results_i3d])
results_i3d_df.to_csv(RESULTS_CSV_PATH, index=False)
results_i3d_df


In [ ]:
cm_i3d = confusion_matrix(test_labels_i3d, test_preds_i3d)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_i3d,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Abnormal"],
    yticklabels=["Normal", "Abnormal"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - MedicalNet ResNet-18")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"intensity_selected_medicalnet_resnet18_top{TOP_K_SLICES}_confusion_matrix.png", dpi=200)
plt.show()

print(classification_report(test_labels_i3d, test_preds_i3d, target_names=["Normal", "Abnormal"]))


In [ ]:
fpr_i3d, tpr_i3d, thresholds_i3d = roc_curve(test_labels_i3d, test_probs_i3d)
roc_auc_i3d = auc(fpr_i3d, tpr_i3d)
threshold_idx = np.abs(thresholds_i3d - BEST_THRESHOLD_I3D).argmin()

plt.figure(figsize=(6, 5))
plt.plot(fpr_i3d, tpr_i3d, label=f"MedicalNet ResNet-18 (AUC = {roc_auc_i3d:.3f})")
plt.scatter(
    fpr_i3d[threshold_idx],
    tpr_i3d[threshold_idx],
    color="red",
    s=70,
    label=f"Selected threshold = {BEST_THRESHOLD_I3D:.2f}",
)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - MedicalNet ResNet-18")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"intensity_selected_medicalnet_resnet18_top{TOP_K_SLICES}_roc_curve.png", dpi=200)
plt.show()

print("ROC-AUC:", roc_auc_i3d)
print("Selected threshold:", BEST_THRESHOLD_I3D)


In [ ]:
# Optional: compare with your earlier recorded results manually.
comparison_df = pd.DataFrame([
    {
        "model": "All-Slices 2D CNN Tuned",
        "accuracy": 0.729306,
        "precision": 0.672646,
        "recall": 0.757576,
        "f1": 0.712589,
        "auc": 0.835118,
        "parameters": "larger than EdgeSliceCNN",
    },
    {
        "model": "Lightweight EdgeSliceCNN",
        "accuracy": 0.765101,
        "precision": 0.781818,
        "recall": 0.651515,
        "f1": 0.710744,
        "auc": 0.825585,
        "parameters": 24225,
    },
    {
        "model": "Previous Denoised 3D CNN",
        "accuracy": 0.727069,
        "precision": 0.688119,
        "recall": 0.702020,
        "f1": 0.695000,
        "auc": 0.793254,
        "parameters": 883601,
    },
    {
        "model": f"Intensity-Selected MedicalNet ResNet-18 Top-{TOP_K_SLICES}",
        "accuracy": results_i3d["accuracy"],
        "precision": results_i3d["precision"],
        "recall": results_i3d["recall"],
        "f1": results_i3d["f1"],
        "auc": results_i3d["auc"],
        "parameters": total_params,
    },
])

comparison_df
